In [1]:
!pip install -q datasets langchain transformers langchain-community

In [2]:
!pip install -q bitsandbytes

In [3]:
import os

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [4]:
from datasets import load_dataset
import torch
import torch.nn as nn
import langchain
import random
from torch.utils.data import Dataset, DataLoader , IterableDataset
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [60]:

class LLM(nn.Module):

  def __init__(
      self,
      vocab_size=tokenizer.vocab_size,
      d_model=768,
      nhead=8,
      num_layers=5,
      max_seq_len=512,
  ):
    super().__init__()
    self.d_model = d_model


    self.tok_embedding = nn.Embedding(vocab_size, d_model)
    self.pos_embedding = nn.Embedding(max_seq_len, d_model)

    encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_model, nhead=nhead, batch_first=True
    )
    self.transformer_encoder = nn.TransformerEncoder(
        encoder_layer, num_layers=num_layers
    )

    decoder_layer = nn.TransformerDecoderLayer(
        d_model=d_model, nhead=nhead, batch_first=True
    )
    self.transformer_decoder = nn.TransformerDecoder(
        decoder_layer, num_layers=num_layers
    )

    self.fc_out = nn.Linear(d_model, vocab_size)

  def embed(self, input_ids):
    """Computes token + scaled positional embeddings."""
    seq_len = input_ids.size(1)
    positions = torch.arange(0, seq_len, device=input_ids.device).unsqueeze(0)
    return (self.tok_embedding(input_ids) * (self.d_model**0.5)) + (
        self.pos_embedding(positions)
    )

  def forward(
      self,
      input_ids_encoder,
      attention_mask_encoder,
      input_ids_decoder,
      attention_mask_decoder,
  ):

    encoder_embedded = self.embed(input_ids_encoder)
    decoder_embedded = self.embed(input_ids_decoder)

    tgt_seq_len = input_ids_decoder.size(1)
    tgt_mask = (
        nn.Transformer.generate_square_subsequent_mask(tgt_seq_len)
        .to(dtype=torch.float32, device=input_ids_decoder.device)
        .masked_fill(
            nn.Transformer.generate_square_subsequent_mask(tgt_seq_len)
            .to(device=input_ids_decoder.device)
            .bool(),
            float("-inf"),
        )
    )


    src_key_padding_mask = (attention_mask_encoder == 0).bool()
    tgt_key_padding_mask = (attention_mask_decoder == 0).bool()

    encoder_output = self.transformer_encoder(
        encoder_embedded, src_key_padding_mask=src_key_padding_mask
    )


    decoder_output = self.transformer_decoder(
        tgt=decoder_embedded,
        memory=encoder_output,
        tgt_mask=tgt_mask,
        tgt_key_padding_mask=tgt_key_padding_mask,
        memory_key_padding_mask=src_key_padding_mask,
    )

    return self.fc_out(decoder_output)


In [ ]:
dataset = load_dataset("wikimedia/wikipedia","20231101.en",streaming=True)
dataset

In [ ]:
len(next(iter(dataset['train']))['text'].split())

In [8]:
def process_stream(stream):
  for example in stream:
    yield { "text": example["text"] }

class IterableData(IterableDataset):
  def __init__(self, tr_data ):
    self.data = tr_data

  def __iter__(self):
    return process_stream(self.data)


In [9]:
iterdata = DataLoader(IterableData(dataset['train']), batch_size=4)

In [11]:
if torch.cuda.is_available():
  device = "cuda"
else:
  device = "cpu"

In [ ]:
import gc
import logging
import warnings
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.getLogger("numba").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)
gc.collect()
torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"


model = LLM(vocab_size=len(tokenizer), max_seq_len=512).to(device)

EPOCHS = 5
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
spec = torch.amp.GradScaler(enabled=True)
gradient_accumulation_steps = 4

bos_id = (
    tokenizer.bos_token_id
    if tokenizer.bos_token_id is not None
    else tokenizer.eos_token_id
)
if bos_id is None:
  bos_id = 50256

writer = SummaryWriter("runs/llm_training_experiment")

for epoch in tqdm(range(EPOCHS), desc="Epochs"):
  print("\nEPOCH : ", epoch + 1)
  epoch_loss = 0.0
  num_batches_processed = 0

  for batch_idx, batch in enumerate(
      tqdm(iterdata, leave=False, desc=f"Epoch {epoch+1} Batches")
  ):
    if batch_idx > 2000:
      break

    input_texts = batch["text"]

    in_data = tokenizer(
        input_texts,
        max_length=512,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    seq_len = in_data["input_ids"].shape[1]
    if seq_len < 4:
      continue

    # Split sequence at 85% mark
    split_point = int(seq_len * 0.85)
    if split_point >= seq_len - 1:
      split_point = seq_len - 2

    input_ids_encoder = in_data["input_ids"][:, :split_point].to(device)
    attention_mask_encoder = in_data["attention_mask"][:, :split_point].to(
        device
    )

    raw_target_ids = in_data["input_ids"][:, split_point:].to(device)
    raw_target_mask = in_data["attention_mask"][:, split_point:].to(device)


    batch_size = raw_target_ids.size(0)
    bos_tensor = torch.full(
        (batch_size, 1), bos_id, device=device, dtype=torch.long
    )
    bos_mask = torch.ones((batch_size, 1), device=device, dtype=torch.long)

    # Shifted sequences: decoder input receives BOS token at index 0
    decoder_input = torch.cat([bos_tensor, raw_target_ids[:, :-1]], dim=1)
    decoder_attention_mask_for_forward = torch.cat(
        [bos_mask, raw_target_mask[:, :-1]], dim=1
    )


    labels = raw_target_ids


    with torch.amp.autocast(device_type=device, dtype=torch.float16):
      result = model(
          input_ids_encoder,
          attention_mask_encoder,
          decoder_input,
          decoder_attention_mask_for_forward,
      )


      result_reshaped = result.reshape(-1, result.size(-1))
      labels_reshaped = labels.reshape(-1)
      loss = loss_fn(result_reshaped, labels_reshaped)

    loss = loss / gradient_accumulation_steps
    epoch_loss += loss.item()
    num_batches_processed += 1

    spec.scale(loss).backward()

    # Step optimizer on gradient accumulation boundaries
    if (batch_idx + 1) % gradient_accumulation_steps == 0:
      spec.step(optimizer)
      spec.update()
      optimizer.zero_grad()

  if (
      num_batches_processed % gradient_accumulation_steps
  ) != 0 and num_batches_processed > 0:
    spec.step(optimizer)
    spec.update()
    optimizer.zero_grad()

  avg_epoch_loss = (
      epoch_loss / num_batches_processed if num_batches_processed > 0 else 0.0
  )
  writer.add_scalar("Loss/train", avg_epoch_loss, epoch)
  print("LOSS : ", avg_epoch_loss)
  print("-" * 25)

writer.close()

In [13]:
from pathlib import Path
doc = Path("model") / "pretrained_llm_weights.pt"
doc.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), doc)

In [ ]:
dolly_dataset = load_dataset("databricks/databricks-dolly-15k")
dolly_dataset

In [ ]:
dolly_dataset['train'][random.randint(0, len(dolly_dataset['train']))]

In [61]:
import gc
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import get_cosine_schedule_with_warmup

gc.collect()
torch.cuda.empty_cache()

bos_id = (
    tokenizer.bos_token_id
    if tokenizer.bos_token_id is not None
    else tokenizer.eos_token_id
)
if bos_id is None:
  bos_id = 50256


class InstructionDataset(Dataset):

  def __init__(self, dataset, tokenizer, max_enc_len=512, max_dec_len=256):
    self.data = list(dataset)
    self.tokenizer = tokenizer
    self.max_enc_len = max_enc_len
    self.max_dec_len = max_dec_len

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    item = self.data[idx]
    instruction = item.get("instruction", "")
    context = item.get("context", "")
    response = item.get("response", "")

    if context:
      prompt = f"Instruction: {instruction}\nContext: {context}\nAnswer:"
    else:
      prompt = f"Instruction: {instruction}\nAnswer:"

    enc = self.tokenizer(
        prompt,
        max_length=self.max_enc_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )


    dec = self.tokenizer(
        response,
        max_length=self.max_dec_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    enc_input_ids = enc["input_ids"].squeeze(0)
    enc_attn_mask = enc["attention_mask"].squeeze(0)

    raw_dec_input_ids = dec["input_ids"].squeeze(0)
    raw_dec_attn_mask = dec["attention_mask"].squeeze(0)


    bos_tensor = torch.tensor([bos_id], dtype=torch.long)
    bos_mask = torch.tensor([1], dtype=torch.long)

    decoder_input_ids = torch.cat([bos_tensor, raw_dec_input_ids[:-1]], dim=0)
    decoder_attn_mask = torch.cat([bos_mask, raw_dec_attn_mask[:-1]], dim=0)


    labels = raw_dec_input_ids.clone()
    labels[labels == self.tokenizer.pad_token_id] = -100

    return {
        "encoder_input_ids": enc_input_ids,
        "encoder_attention_mask": enc_attn_mask,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_attn_mask,
        "labels": labels,
    }



In [62]:

dataset_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
fine_tune_loader = DataLoader(
    InstructionDataset(dataset_dolly, tokenizer), batch_size=4, shuffle=True
)


In [ ]:

fine_tuned_model = LLM(vocab_size=len(tokenizer), max_seq_len=512)


fine_tuned_model.load_state_dict(torch.load("model/pretrained_llm_weights.pt"))
fine_tuned_model.to(device)


In [64]:

optimizer = torch.optim.AdamW(fine_tuned_model.parameters(), lr=2e-5)


loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100
)

In [ ]:

fine_tuned_model.train()

FINE_TUNE_EPOCHS = 3
gradient_accumulation_steps = 4

spec = torch.amp.GradScaler(enabled=True)

total_steps = (
    len(fine_tune_loader) // gradient_accumulation_steps
) * FINE_TUNE_EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)



for epoch in range(FINE_TUNE_EPOCHS):
  print(f"\n--- FINE-TUNING EPOCH {epoch + 1}/{FINE_TUNE_EPOCHS} ---")
  epoch_loss = 0.0
  num_batches = 0

  for batch_idx, batch in enumerate(
      tqdm(fine_tune_loader, desc=f"Epoch {epoch + 1}")
  ):
    enc_ids = batch["encoder_input_ids"].to(device)
    enc_mask = batch["encoder_attention_mask"].to(device)
    dec_ids = batch["decoder_input_ids"].to(device)
    dec_mask = batch["decoder_attention_mask"].to(device)
    labels = batch["labels"].to(device)


    with torch.amp.autocast(device_type=device, dtype=torch.float16):
      outputs = fine_tuned_model(enc_ids, enc_mask, dec_ids, dec_mask)

      logits_reshaped = outputs.reshape(-1, outputs.size(-1))
      labels_reshaped = labels.reshape(-1)
      loss = loss_fn(logits_reshaped, labels_reshaped)

    loss = loss / gradient_accumulation_steps
    epoch_loss += loss.item() * gradient_accumulation_steps
    num_batches += 1

    spec.scale(loss).backward()

    if (batch_idx + 1) % gradient_accumulation_steps == 0:
      spec.step(optimizer)
      scheduler.step()
      spec.update()
      optimizer.zero_grad()

  if (num_batches % gradient_accumulation_steps) != 0 and num_batches > 0:
    spec.step(optimizer)
    scheduler.step()
    spec.update()
    optimizer.zero_grad()

  avg_loss = epoch_loss / num_batches if num_batches > 0 else 0.0
  print(f"FINE-TUNING EPOCH {epoch + 1} LOSS: {avg_loss:.4f}")

In [ ]:
!pip install -q torchinfo
import torchinfo
torchinfo.summary(model)

In [ ]:
torchinfo.summary(fine_tuned_model)

In [68]:
Fine_Tuned_Doc = Path("model") / "fine_tuned_llm_weights.pt"
Fine_Tuned_Doc.parent.mkdir(parents=True, exist_ok=True)
torch.save(fine_tuned_model.state_dict(), Fine_Tuned_Doc)

In [ ]:
!pip install -q langchain_text_splitters youtube-transcript-api langchain-huggingface chromadb==0.5.0 opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1

In [70]:
!pip install -q langchain-chroma

In [71]:
@torch.no_grad()
def generate_response_text(
    fine_tuned_model,
    tokenizer,
    encoder_input_ids,
    encoder_attention_mask,
    max_length=100,
    repetition_penalty=1.5,
):
  fine_tuned_model.eval()
  device = encoder_input_ids.device
  batch_size = encoder_input_ids.size(0)

  start_token_id = tokenizer.bos_token_id or 50256
  decoder_input_ids = torch.full(
      (batch_size, 1), start_token_id, device=device, dtype=torch.long
  )

  pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -1
  comma_id = tokenizer.encode(",")[0]

  for step in range(max_length):
    decoder_attn_mask = torch.ones_like(decoder_input_ids, device=device)

    logits = fine_tuned_model(
        encoder_input_ids,
        encoder_attention_mask,
        decoder_input_ids,
        decoder_attn_mask,
    )

    next_token_logits = logits[:, -1, :].clone()

    for token_id in set(decoder_input_ids[0].tolist()):
      if next_token_logits[0, token_id] < 0:
        next_token_logits[0, token_id] *= repetition_penalty
      else:
        next_token_logits[0, token_id] /= repetition_penalty

    if pad_id != -1:
      next_token_logits[:, pad_id] = -float("inf")
    if step == 0:
      next_token_logits[:, comma_id] = -float("inf")

    next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
    decoder_input_ids = torch.cat([decoder_input_ids, next_token_id], dim=-1)


    if (
        step > 5
        and tokenizer.eos_token_id is not None
        and (next_token_id == tokenizer.eos_token_id).all()
    ):
      break


  return tokenizer.decode(
      decoder_input_ids[0],
      skip_special_tokens=True,
      clean_up_tokenization_spaces=False,
  ).strip()


def LLM_Assistant_Youtube(url: str, query: str, use_rag: bool = False) -> str:
  """Fetches YouTube video transcript and generates an answer using fine_tuned_model."""
  try:
    loader = YoutubeLoader.from_youtube_url(url, add_video_info=False)
    docs = loader.load()
    context_text = (
        " ".join([doc.page_content for doc in docs])
        if docs
        else "No transcript available."
    )
  except Exception as e:
    context_text = f"Could not retrieve transcript: {str(e)}"

  context_text = context_text[:1200]
  prompt = f"Instruction: {query}\nContext: {context_text}\nAnswer:"

  enc = tokenizer(
      prompt,
      max_length=512,
      truncation=True,
      padding=True,
      return_tensors="pt",
  ).to(device)

  generated_answer = generate_response_text(
      fine_tuned_model,
      tokenizer,
      enc["input_ids"],
      enc["attention_mask"],
      max_length=100,
  )

  return generated_answer

In [ ]:
URL = "https://youtu.be/msFxQ7OYPj8?si=Kbpz9Y-PJmFFhaUB"
result = LLM_Assistant_Youtube(URL,"how to learn AI from scratch")
print(f"Generated Answer: {result}")